# Choosing (and validating) the calibration data

Notebooks 11-13 showed every *spectral* subset trick is null — the training **population** (lot
248+251) is the lever. So instead of hunting for a cleverer subset, this notebook does the two
things that actually matter for a defensible calibration: **quality-filter by provenance**, and
**validate that the data is the right choice** (coverage + transfer). All from the local mirror.

- **A. Provenance** — drop filters flagged for a *physical* problem (mass/contamination/damage),
  using the flag history reconstructed from comment text (`status_events.csv`).
- **B. Coverage** — is ETAD *inside* the training spectral domain, or extrapolation?
- **C. Transfer** — leave-one-**site**-out RMSE (Addis is an unseen site).
- **D. Designed subsets** — Kennard-Stone / EC-stratified vs just using everything.

In [1]:
from pathlib import Path
import sys, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import cross_val_predict, KFold

plt.rcParams.update({"axes.facecolor": "white", "figure.facecolor": "white", "axes.grid": True, "grid.color": "0.9"})
LOCAL = Path.home() / "Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/FTIR/local_db"
GD = Path.home() / ("Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/University"
                    "/Research/Grad/UC Davis Ann/NASA MAIA/Data")
Path("figures").mkdir(exist_ok=True); Path("tables").mkdir(exist_ok=True)

sys.path.insert(0, str(LOCAL)); import local_calib
df, wn = local_calib.assemble("EC"); df = df[df["y"] > 0].reset_index(drop=True)
Ximp = df[wn].to_numpy(float); yimp = df["y"].to_numpy(float); WN = np.array([float(c) for c in wn])
sites = df["Site"].to_numpy(); fids = df["FilterId"].to_numpy()

ET = GD / "DAVIS/ETAD FTIR"; spec = pd.read_csv(ET / "ETAD_FTIR_spectra.csv"); meta = pd.read_csv(ET / "ETAD_metadata.csv")
wc = sorted([c for c in spec.columns if c not in ("SampleAnalysisId", "MediaId")], key=lambda c: -float(c))
Xeth = spec[wc].to_numpy(float); media = spec["MediaId"].to_numpy()
vol = pd.Series(media).map(meta.set_index("MediaId")["SampleVolume_m3"]).to_numpy(); volok = np.where(np.isfinite(vol) & (vol > 0), vol, np.nan)
hips = pd.read_csv(GD / "Spartan/SPARTAN_HIPS_Batch1-51.v2.csv", usecols=["Site", "FilterId", "Fabs"])
hips = hips[hips["Site"] == "ETAD"].rename(columns={"FilterId": "ExternalFilterId"}); m2 = meta[["MediaId", "ExternalFilterId"]]

def hf(load):
    d = pd.DataFrame({"MediaId": media, "c": load / volok}).merge(m2, on="MediaId").merge(hips, on="ExternalFilterId")
    d["ec"] = d["Fabs"] / 10.0; d = d[np.isfinite(d["ec"]) & np.isfinite(d["c"])]
    s, b = np.polyfit(d["ec"], d["c"], 1); r2 = 1 - np.sum((d["c"]-(s*d["ec"]+b))**2)/np.sum((d["c"]-d["c"].mean())**2)
    return dict(slope=float(s), intercept=float(b), r2=float(r2))
def etad(mask, k=10):
    return hf(PLSRegression(k, scale=False).fit(Ximp[mask], yimp[mask]).predict(Xeth).ravel())
allm = np.ones(len(yimp), bool)
print(f"train {Ximp.shape}  sites={df.Site.nunique()}  EC median {np.median(yimp):.1f}")

/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/FTIR/local_db/local_calib.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  spec["date"] = pd.to_datetime(spec["SampleDate"], errors="coerce").dt.date


/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/FTIR/local_db/local_calib.py:22: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  r = pd.read_csv(LOCAL / "tables" / f"results_{_TABLE[analyte]}.csv")


/Users/ahmadjalil/Library/CloudStorage/GoogleDrive-ahzs645@gmail.com/My Drive/FTIR/local_db/local_calib.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  r["date"] = pd.to_datetime(r["SampleDate"], errors="coerce").dt.date


train (13012, 2722)  sites=158  EC median 3.8


## A. Provenance quality filter — physical vs administrative flags

`status_events.csv` reconstructs each filter's flag history from operator comments. A filter
flagged for a **physical** problem is a bad *measurement* and should leave the calibration even if
its current status is NM; an administrative flag-then-revalidate is fine to keep.

In [2]:
ev = pd.read_csv(LOCAL / "status_events.csv")
PHYS = r"mass|weight|negativ|contamin|damag|tear|hole|punctur|dropp|broke|crack|spill|leak|rip|fell|scratch|smudge|fold|wrinkl"
ev["phys"] = ev["Entry"].astype(str).str.lower().str.contains(PHYS) & (ev["event"] == "flag")
phys_fids = set(ev.loc[ev["phys"], "FilterId"])
bad = np.array([f in phys_fids for f in fids])
print(f"training filters ever physically-flagged: {bad.sum()} of {len(fids)} ({100*bad.mean():.1f}%)")
print("\nexample physical-flag comments:")
for e in ev.loc[ev["phys"], "Entry"].astype(str).head(3): print("  -", e[:110])
print()
for label, m in [("full set", allm), ("drop physical-flagged", ~bad)]:
    f = etad(m); print(f"  {label:22s} n={int(m.sum()):5d}: slope={f['slope']:.2f} "
                       f"int={f['intercept']:+.2f} r2={f['r2']:.2f}")

training filters ever physically-flagged: 108 of 13012 (0.8%)

example physical-flag comments:
  - Issue with mass of filter, the filter weight difference was a negative number, which should not happen. Pre-we
  - Issue with mass of filter, the filter weight difference was a negative number, which should not happen. Pre-we
  - Issue with mass of filter, the filter weight difference was a negative number which should not happen. Pre-wei



  full set               n=13012: slope=1.21 int=-2.29 r2=0.67


  drop physical-flagged  n=12904: slope=1.19 int=-2.24 r2=0.67


## B. Coverage — is ETAD inside the training spectral domain?

Leverage = whitened PCA distance to the training centroid. If ETAD sits below the training 99th
percentile, the predictions are **interpolation**, not extrapolation — a direct test of whether the
lot-248+251 data is the right population.

In [3]:
mu = Ximp.mean(0); P = PCA(10, random_state=0).fit(Ximp - mu); w = 1/np.sqrt(P.explained_variance_)
dtr = np.sqrt(((P.transform(Ximp - mu)*w - (P.transform(Ximp - mu)*w).mean(0))**2).sum(1))
cen = (P.transform(Ximp - mu)*w).mean(0)
det = np.sqrt(((P.transform(Xeth - mu)*w - cen)**2).sum(1))
thr = np.percentile(dtr, 99)
print(f"ETAD beyond training 99th-pct leverage: {100*(det > thr).mean():.0f}%  "
      f"(ETAD median leverage {np.median(det):.1f} vs training {np.median(dtr):.1f})")

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.hist(dtr, bins=60, alpha=0.6, density=True, label=f"IMPROVE lot 248+251 (n={len(dtr)})", color="#1f77b4")
ax.hist(det, bins=30, alpha=0.6, density=True, label=f"ETAD / Addis (n={len(det)})", color="#d62728")
ax.axvline(thr, color="k", ls="--", lw=1, label="training 99th pct")
ax.set_xlabel("leverage (whitened PCA distance to training centre)"); ax.set_ylabel("density")
ax.set_title("ETAD sits INSIDE the lot-248+251 training domain → interpolation, not extrapolation")
ax.set_xlim(0, np.percentile(np.r_[dtr, det], 99.5)); ax.legend()
plt.tight_layout(); plt.savefig("figures/fig14_etad_coverage_leverage.png", dpi=140, bbox_inches="tight")
print("saved figures/fig14_etad_coverage_leverage.png"); plt.show()

ETAD beyond training 99th-pct leverage: 0%  (ETAD median leverage 3.0 vs training 1.9)


saved figures/fig14_etad_coverage_leverage.png


/var/folders/7k/65ckdzsj0w3171qv80rh8dgr0000gn/T/ipykernel_61263/1886884694.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  print("saved figures/fig14_etad_coverage_leverage.png"); plt.show()


## C. Leave-one-SITE-out — the honest transfer cost to a new site

Random CV shares sites across folds (optimistic). Holding out **whole sites** estimates transfer to
an unseen site — which is exactly what applying to Addis is.

In [4]:
def rmse_loso(k=10):
    pred = np.full(len(yimp), np.nan)
    for st in np.unique(sites):
        te = sites == st
        m = PLSRegression(k, scale=False).fit(Ximp[~te], yimp[~te]); pred[te] = m.predict(Ximp[te]).ravel()
    return pred
loso = rmse_loso()
rp = cross_val_predict(PLSRegression(10, scale=False), Ximp, yimp, cv=KFold(5, shuffle=True, random_state=0)).ravel()
r_rand = np.sqrt(np.mean((yimp - rp)**2)); r_loso = np.sqrt(np.nanmean((yimp - loso)**2))
print(f"random 5-fold RMSE:      {r_rand:.2f} µg")
print(f"leave-one-site-out RMSE: {r_loso:.2f} µg  (+{100*(r_loso/r_rand-1):.0f}% — the transfer cost)")
print("→ small penalty: the calibration generalises across sites, so applying to Addis is defensible.")

random 5-fold RMSE:      10.18 µg
leave-one-site-out RMSE: 10.65 µg  (+5% — the transfer cost)
→ small penalty: the calibration generalises across sites, so applying to Addis is defensible.


## D. Designed subsets — Kennard-Stone / EC-stratified vs the full set

In [5]:
rng = np.random.default_rng(0)
S = P.transform(Ximp - mu) * w
def ks(N):
    sel = [int(np.argmax(((S - S.mean(0))**2).sum(1)))]; mind = ((S - S[sel[0]])**2).sum(1)
    for _ in range(N - 1):
        i = int(np.argmax(mind)); sel.append(i); mind = np.minimum(mind, ((S - S[i])**2).sum(1))
    m = np.zeros(len(yimp), bool); m[sel] = True; return m
def strat(N):
    b = np.clip((yimp / (yimp.max()/10)).astype(int), 0, 9); idx = []
    for k in range(10):
        c = np.where(b == k)[0]; idx += list(rng.choice(c, min(len(c), N//10), replace=False))
    m = np.zeros(len(yimp), bool); m[idx] = True; return m
rm = np.zeros(len(yimp), bool); rm[rng.choice(len(yimp), 2000, replace=False)] = True

rows = [{"selection": "full set", "n": int(allm.sum()), **etad(allm)}]
for nm, m in [("random 2000", rm), ("EC-stratified", strat(2000)), ("Kennard-Stone 2000", ks(2000))]:
    rows.append({"selection": nm, "n": int(m.sum()), **etad(m)})
tbl = pd.DataFrame(rows).round(3); tbl.to_csv("tables/calibration_data_selection_summary.csv", index=False)
print(tbl.to_string(index=False))

         selection     n  slope  intercept    r2
          full set 13012  1.208     -2.295 0.674
       random 2000  2000  1.519     -2.903 0.639
     EC-stratified   316  2.140     -4.213 0.649
Kennard-Stone 2000  2000  1.308     -2.677 0.660


### Recommendation — how to choose the calibration data
- **Use the full lot-248+251 set, quality-filtered** (drop the ~108 physically-flagged filters).
  Every *clever* subset — PCA neighborhood, biomass-ratio, EC-range, Kennard-Stone, EC-stratified —
  is null-or-worse. EC-stratifying is actively harmful (it up-weights the rare extreme-EC filters
  that steepen the slope).
- **It's validated, not guessed:** ETAD sits **inside** the training spectral domain (0%
  extrapolation), and the calibration **transfers across sites** (leave-one-site-out costs only
  ~5% RMSE) — so applying it to Addis is defensible interpolation.
- **The real levers are quality + validation, not subset cleverness.** Provenance-filter for genuine
  measurement problems; prove coverage (leverage) and transfer (leave-one-site-out); then use the
  representative population. That is a defensible, publishable data-selection story.
- Still no Ethiopia TOR truth — HIPS agreement + physical sensibility remain the only external checks.